

To install environment, `conda env create -f environment-HGQ.yml`, or to update env `conda env update -f environment-HGQ.yml`. Remember to restart kernel.

In [1]:
import os
model_to_test = 'MNIST_CNN'
model_revision = '4'
hls4ml_revision = 'mnist_hls4ml_VU'

base_dir = os.path.abspath(model_to_test)
model_dir = os.path.join(base_dir, model_revision)
os.makedirs(model_dir, exist_ok=True)

description = f"""
# Model Configuration


- **Model architecture description**: {model_to_test}
- **Model Revision**: {model_revision}
- **HLS4ML Revision**: {hls4ml_revision}
- **Target Device**: KV260 (xck26-sfvc784-2LV-c)
- **Dataset**: HLS4ML LHC Jets
- **Vivado/Vitis**: 2025.2
"""
output_dir = os.path.join(model_dir, f"hls4ml_prj_{hls4ml_revision}")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
    f.write(description)

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import os
from sklearn.metrics import accuracy_score

%matplotlib inline
seed = 0
np.random.seed(seed)

tf.random.set_seed(seed)

2026-05-05 11:52:22.093469: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']
!vitis --version
!vitis_hls -version
!vivado -version


****** Vitis Development Environment
****** Vitis v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:14
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

/bin/bash: line 1: vitis_hls: command not found
vivado v2025.2 (64-bit)
Tool Version Limit: 2025.11
SW Build 6299465 on Fri Nov 14 12:34:56 MST 2025
IP Build 6300035 on Fri Nov 14 10:48:45 MST 2025
SharedData Build 6298862 on Thu Nov 13 04:50:51 MST 2025
Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.


In [37]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_test = X_test.astype("float32") / 255
y_test = keras.utils.to_categorical(y_test, 10)

In [4]:
import scipy.io
import numpy as np

def load_svhn(path):
    data = scipy.io.loadmat(path)
    X = data["X"]
    y = data["y"].reshape(-1)

    X = np.transpose(X, (3,0,1,2))

    y[y == 10] = 0

    return X, y

X_test, y_test   = load_svhn("/home/ncgadmin/DAT255/DAT255-project/SVHN/test_32x32.mat")

X_test  = X_test.astype("float32") / 255.0
y_test  = keras.utils.to_categorical(y_test, 10)

In [5]:
# Prepare subset of testdata for simulation (running everything takes an unnecessary long time)
simulation_rows = 100
x_test_sim_path = os.path.join(base_dir, "x_test_sim.npy")
y_test_sim_path = os.path.join(base_dir, "y_test_sim.npy")
np.save(x_test_sim_path, X_test[:simulation_rows])
np.save(y_test_sim_path, y_test[:simulation_rows])

Load existing model

In [5]:
from keras.models import load_model
from qkeras.utils import load_qmodel
#import hgq.layers

#keras_model_path = os.path.join(base_dir, "/home/ncgadmin/DAT255/DAT255-project/MNIST/CNN_HGQ_StaticTraining/test_models/epoch=2619-val_acc=0.974-ebops=48591-val_loss=0.091.keras")
#keras_model_path = os.path.join(base_dir, "/home/ncgadmin/DAT255/DAT255-project/MNIST/CNN_HGQ_StaticTraining/test_models/epoch=3277-val_acc=0.967-ebops=30992-val_loss=0.120.keras")
keras_model_path = os.path.join(base_dir, "/home/ncgadmin/DAT255/DAT255-project/SVHN/Qkeras/Model_Qkeras_NoBatchNorm.h5")
model = load_qmodel(keras_model_path)
score = model.evaluate(X_test, y_test)

/home/ncgadmin/miniconda3/envs/devenv-qkeras/lib/python3.10/site-packages/keras/src/initializers/initializers.py:120: UserWarning: The initializer LecunUniform is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


814/814 [==============================] - 5s 6ms/step - loss: 0.4883 - accuracy: 0.9219


In [39]:
from hgq.utils import trace_minmax

trace_minmax(model, X_test, verbose=True)

conv0 : 5408
conv1 : 19068
dense0: 7872
Total: 32348


32348

In [6]:
# Save the model summary to a text file
with open(os.path.join(model_dir, "summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 32, 32, 3)]       0         
                                                                 
 sequential (Sequential)     (None, 32, 32, 3)         0         
                                                                 
 qconv0 (QConv2D)            (None, 30, 30, 32)        896       
                                                                 
 relu0 (QActivation)         (None, 30, 30, 32)        0         
                                                                 
 pool0 (MaxPooling2D)        (None, 15, 15, 32)        0         
                                                                 
 qconv1 (QConv2D)            (None, 13, 13, 64)        18496     
                                                                 
 relu1 (QActivation)         (None, 13, 13, 64)        0     

In [7]:
import keras

inputs = keras.Input(shape=(32, 32, 3), name="hw_input")

x = model.get_layer('qconv0')(inputs)
x = model.get_layer('relu0')(x)
x = model.get_layer('pool0')(x)

x = model.get_layer('qconv1')(x)
x = model.get_layer('relu1')(x)
x = model.get_layer('pool1')(x)

x = model.get_layer('qconv2')(x)
x = model.get_layer('relu2')(x)
x = model.get_layer('pool2')(x)

x = keras.layers.Flatten()(x)

x = model.get_layer('qdense0')(x)
x = model.get_layer('relu3')(x)

x = model.get_layer('dropout')(x) 

x = model.get_layer('qdense1')(x)
outputs = model.get_layer('softmax')(x)


model_stripped = keras.Model(inputs=inputs, outputs=outputs)

model_stripped.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hw_input (InputLayer)       [(None, 32, 32, 3)]       0         
                                                                 
 qconv0 (QConv2D)            (None, 30, 30, 32)        896       
                                                                 
 relu0 (QActivation)         (None, 30, 30, 32)        0         
                                                                 
 pool0 (MaxPooling2D)        (None, 15, 15, 32)        0         
                                                                 
 qconv1 (QConv2D)            (None, 13, 13, 64)        18496     
                                                                 
 relu1 (QActivation)         (None, 13, 13, 64)        0         
                                                                 
 pool1 (MaxPooling2D)        (None, 6, 6, 64)          0     

# Convert and synthesize with HLS4ML
Configure parameters.
KV260: xck26-sfvc784-2LV-c 

In [10]:
import hls4ml
import plotting

config = hls4ml.utils.config_from_keras_model(model_stripped, granularity='name', backend='vitisunified')
config['Model']['ReuseFactor'] = 1
#config['Model']['Strategy'] = 'Resource'
#config['Model']['Strategy'] = 'Distributed Arithmetic'

proj_name = f"{str(model_to_test)}_{str(model_revision)}_hls4ml_prj_{str(hls4ml_revision)}"

print("-----------------------------------")
print("Configuration")
plotting.print_dict(config)
print("-----------------------------------")

hls_model = hls4ml.converters.convert_from_keras_model(
    model_stripped,    
    backend='vitisunified',
    hls_config=config,
    io_type='io_stream',
    proj_name = proj_name,
    output_dir=output_dir, 
    board       = 'kv260',
    part='xck26-sfvc784-2LV-c',
    clock_period='5',
)

hls_model.compile()
#hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True,to_file=os.path.join(output_dir, "model-plot.png"))

-----------------------------------
Configuration
Model
  Precision
    default:         fixed<16,6>
  ReuseFactor:       1
  Strategy:          Latency
  BramFactor:        1000000000
  TraceOutput:       False
LayerName
  hw_input
    Trace:           False
    Precision
      result:        auto
  qconv0
    Trace:           False
    Precision
      result:        auto
      weight:        fixed<4,1,TRN,WRAP,0>
      bias:          fixed<4,1,TRN,WRAP,0>
      accum:         auto
    ReuseFactor:     1
    ParallelizationFactor:1
    ConvImplementation:LineBuffer
  qconv0_linear
    Trace:           False
    Precision
      result:        auto
      table:         fixed<18,8,TRN,WRAP,0>
    ReuseFactor:     1
    TableSize:       1024
  relu0
    Trace:           False
    Precision
      result:        ufixed<4,0,RND_CONV,SAT,0>
      table:         fixed<18,8,TRN,WRAP,0>
    ReuseFactor:     1
    TableSize:       1024
  pool0
    Trace:           False
    Precision
      result

Check performance

In [18]:
y_keras = model.predict(X_test)
y_hls = hls_model.predict(np.ascontiguousarray(X_test))

print("Difference in inference-calculations between Keras-model and HLS4ML-compiled model (first rows):")
for x,y in enumerate(y_keras[:5]):
    print(f"{y-y_hls[x]}")

print("Keras  Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))



334/814 [===========>..................] - ETA: 3s

KeyboardInterrupt: 

In [19]:
max_diff = np.max(np.abs(y_keras - y_hls))
print("Max Bit-Error: {:.6f}".format(max_diff))

NameError: name 'y_hls' is not defined

In [ ]:
hls_model.build(
    csim=False,
    synth=True, 
    bitfile=False
    ) 


****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Tue May  5 11:56:01 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
INFO: [HLS 200-2005] Using work_dir /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/MNIST_CNN/4/hls4ml_prj_mnist_hls4ml_VU/vitis_workspace/myproject/vitis_unified_project 
INFO: [HLS 200-2176] Writing Vitis IDE component file /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/MNIST_CNN/4/hls4ml_prj_mnist_hls4ml_VU/vitis_workspace/myproject/vitis_unified_project/vitis-comp.json
INFO: [HLS 200-10] Creating and opening component '/home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/MNIST_CNN/4/hls4ml_prj_mnist_hls4ml_VU/vitis_workspace/myproject/vitis_unified_project'.
INFO: [HLS 200-1505] Using default flow_target 'vivado'
Resolution: For help on HLS 200-1505 see docs

In [28]:
hls4ml.report.read_vivado_report(os.path.join(output_dir))

Unable to read project data. Exiting.


# Simulation
[vitis unified tutorial](https://github.com/Tanawin1701d/vitis_unified_backend_tutorial/blob/master/03_co_simulation.ipynb)

Ubuntu 24 not supported compiling files for co-sim (glibc missmatch). Using 22-docker for running the cosim.

```bash
sudo docker run  -it -d -p 8888:8888 --name pyct -v /home/ncgadmin/Bachelor:/workspace   -v /tools/Xilinx:/tools/Xilinx   -w /workspace   ubuntu:22.04 bash

sudo docker exec -ti pyct bash

apt update

# Vitis installasjongreier
apt-get install -y locales
sed -i 's/^# *en_US.UTF-8 UTF-8/en_US.UTF-8 UTF-8/' /etc/locale.gen
locale-gen
update-locale LANG=en_US.UTF-8 LC_ALL=en_US.UTF-8

bash /tools/Xilinx/Vitis/2023.2/scripts/installLibs.sh

# Python greier
apt install python3-pip python3 git

curl -O https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh

bash Miniconda3-latest-Linux-x86_64.sh


curl -O https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh

cd HLS4ML_testbench_KV260/development/

source /tools/Xilinx/Vitis_HLS/2023.2/settings64.sh
source /tools/Xilinx/Vivado/2023.2/settings64.sh
source /tools/Xilinx/Model_Composer/2023.2/settings64.sh
source /tools/Xilinx/Vitis/2023.2/settings64.sh

conda activate devenv-vu
jupyter notebook --allow-root --ip=0.0.0.0


In [30]:
# Build and do co-simulation
hls_model.build(
    synth=True, # Only needs to run first time
    cosim=True,
    ) 

# Problem 1 19.03.2026
# Version missmatch mellom system glibc og Vitis 2023.2 (binutils)
# Kjøre i Ubuntu 22 docker, se over


****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Sun Mar 29 18:16:51 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
INFO: [HLS 200-2005] Using work_dir /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2_fixed/1/hls4ml_prj_fix_test_hgq2/vitis_workspace/myproject/vitis_unified_project 
INFO: [HLS 200-2176] Writing Vitis IDE component file /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2_fixed/1/hls4ml_prj_fix_test_hgq2/vitis_workspace/myproject/vitis_unified_project/vitis-comp.json
INFO: [HLS 200-10] Creating and opening component '/home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2_fixed/1/hls4ml_prj_fix_test_hgq2/vitis_workspace/myproject/vitis_unified_project'.
INFO: [HLS 200-1505] Using default flow_target 'vivado'
Resolution: For help on HLS 200-1505 see docs.am

In [32]:
y_baseline = hls_model.predict(np.ascontiguousarray(X_test[:simulation_rows]))
#y_baseline = model.predict(np.ascontiguousarray(X_test[:simulation_rows]))
y_simulation = np.loadtxt(os.path.join(output_dir, "tb_data/rtl_cosim_results.log"))

In [33]:
print(f"y_baseline shape: {y_baseline.shape} and y_simulation: {y_simulation.shape}")
#print(y_simulation)

y_baseline shape: (100, 5) and y_simulation: (100, 5)


In [34]:
assert np.allclose(y_baseline, y_simulation, rtol=0.0, atol=1e-4), (
    "The results from bridge and cosim are NOT equal!"
)
print("\n✅ RTL co-simulation comparison passed, absolute difference is less than 1e-4 (atol=1e-4).")


✅ RTL co-simulation comparison passed, absolute difference is less than 1e-4 (atol=1e-4).


In [35]:
from sklearn.metrics import accuracy_score
print("Difference in inference-calculations between HLS4ML bridge and simulated inference (first rows):")
abs_diff = y_baseline[:3] - y_simulation[:3]
print(np.round(abs_diff, 8))

mse = np.mean(np.square(y_baseline - y_simulation))
print(f"MSE: {mse}")


print("Baseline  Accuracy: {}".format(accuracy_score(np.argmax(y_test[:simulation_rows], axis=1), np.argmax(y_baseline, axis=1))))
print("Simulation Accuracy: {}".format(accuracy_score(np.argmax(y_test[:simulation_rows], axis=1), np.argmax(y_simulation, axis=1))))

Difference in inference-calculations between HLS4ML bridge and simulated inference (first rows):
[[-3.75e-06  3.75e-06  0.00e+00  0.00e+00  0.00e+00]
 [-1.25e-06 -3.75e-06  5.00e-06  0.00e+00  0.00e+00]
 [ 2.50e-07 -5.00e-07 -2.50e-06  0.00e+00  0.00e+00]]
MSE: 1.3737894999784499e-11
Baseline  Accuracy: 0.77
Simulation Accuracy: 0.77
